# Pipeline dos Dados

In [1]:
import pandas as pd
import numpy as np
import math
from collections import defaultdict
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [2]:
PROCESSED = Path('../../data/processed')
df_edges = pd.read_csv(PROCESSED / 'edges.csv')
print(f"Dados reais carregados! Total de arestas com contexto: {len(df_edges)}")
display(df_edges.head())

Dados reais carregados! Total de arestas com contexto: 821


,edge_id,case_id,source_label,target_label,relation,contexto
0,E_001,PMC10106591_01,female,intermittent chest pain,HAS_SYMPTOM,A 44-year-old female patient presented to the ...
1,E_002,PMC10106591_01,physical examination,admission,SUPPORTS,upon
2,E_003,PMC10106591_01,pulmonary examination,normal heart,SUPPORTS,revealed
3,E_004,PMC10106591_01,female,electrocardiogram,UNDERWENT_PROCEDURE,The
4,E_005,PMC10106591_01,female,ecg,UNDERWENT_PROCEDURE,The electrocardiogram (


## Preparando Weak Supervision e Gold Standard

In [4]:
df_edges['contexto'] = df_edges['contexto'].fillna('[ADJACENTE]')
df_treino_ruidoso, df_teste_gold = train_test_split(df_edges, test_size=0.15, random_state=42)

# EXPORTA PARA ANOTAÇÃO MANUAL!
df_export = df_teste_gold[['case_id', 'source_label', 'target_label', 'contexto', 'relation']].copy()
df_export['relation_corrigida_manualmente'] = '' # A equipe preenche isso no Excel!
df_export.to_csv(PROCESSED / 'para_anotar_gold_standard.csv', index=False)

print(f"Treino com regras (Ruído): {len(df_treino_ruidoso)} amostras.")
print(f"Teste (Gold Standard) exportado: {len(df_teste_gold)} amostras.")

Treino com regras (Ruído): 697 amostras.
Teste (Gold Standard) exportado: 124 amostras.


## TF-IDF e DataLoader PyTorch

In [7]:
#df_teste_gold = pd.read_csv(PROCESSED / 'gold_standard_anotado.csv')
df_teste_gold = pd.read_csv(PROCESSED / 'para_anotar_gold_standard.csv')

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=2000)
X_treino_tfidf = vectorizer.fit_transform(df_treino_ruidoso['contexto'])
X_teste_tfidf = vectorizer.transform(df_teste_gold['contexto'])

labels_unicas = df_edges['relation'].unique()
label2id = {label: idx for idx, label in enumerate(labels_unicas)}
id2label = {idx: label for label, idx in label2id.items()}

class MLPDataset(Dataset):
    def __init__(self, matriz_tfidf, labels):
        self.x = torch.tensor(matriz_tfidf.toarray(), dtype=torch.float32)
        self.y = torch.tensor([label2id.get(l, 0) for l in labels], dtype=torch.long)
        
    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

treino_dataset = MLPDataset(X_treino_tfidf, df_treino_ruidoso['relation'].values)
treino_loader = DataLoader(treino_dataset, batch_size=32, shuffle=True)

# NOTA IMPORTANTE: Quando o Excel for preenchido pela equipe, alterar 'relation' para a coluna manual e ler o arquivo novo!
teste_dataset = MLPDataset(X_teste_tfidf, df_teste_gold['relation_corrigida_manualmente'].values)
teste_loader = DataLoader(teste_dataset, batch_size=32, shuffle=False)

print("Tudo pronto! Pipeline de Machine Learning pode iniciar.")
batch_x, batch_y = next(iter(treino_loader))
print(f"Tensores de Entrada X: {batch_x.shape}")
print(f"Tensores de Label Y: {batch_y.shape}")

Tudo pronto! Pipeline de Machine Learning pode iniciar.
Tensores de Entrada X: torch.Size([32, 2000])
Tensores de Label Y: torch.Size([32])
